# P4 HWPX 125 검색 퀵체크 / 세분화 실험 노트북

이 노트북은 `P4 HWPX 125` corpus를 Chroma에 적재하고, KoE5 기반 검색 품질을 확인하기 위한 실행용 노트북입니다.

## 실행 목적

- `outputs/parsing_p4_hwpx_125_0521/chunks_v2_125.jsonl`을 읽습니다. 파일명은 기존과 같고 폴더명으로만 버전을 분리합니다.
- `embed_enabled == true`인 청크만 Chroma에 적재합니다.
- `nlpai-lab/KoE5` 임베딩 모델로 dense 검색을 수행합니다.
- `data/eval/*.csv`의 질문과 정답 문서를 기준으로 검색 결과를 평가합니다.
- `RUN_MODE`만 바꿔서 `smoke`, `quick`, `exp100`, `full` 실행을 선택할 수 있습니다.
- 이 0521 사본은 `RUN_TAG="0521"`을 사용해 결과를 `exp100_0521` 폴더에 저장합니다. 기존 `exp100` 결과와 섞이지 않습니다.

## 실행 모드

- `smoke`: 앞쪽 청크 1,000개만 적재하고 질문 5개만 테스트합니다. 연결과 기본 동작 확인용입니다.
- `quick`: 전체 청크를 적재하고 질문 30개를 dense 기준선으로 빠르게 테스트합니다.
- `exp100`: 전체 청크를 적재하고 100문항 고정 샘플로 기준선 + 5개 검색 조건을 비교합니다.
- `full`: 전체 청크를 적재하고 가능한 eval 질문 전체를 테스트합니다.

## exp100 실험 조건

- `J0_dense_baseline`: KoE5 dense 단독 베이스라인
- `J1_dense_wide`: dense 후보 수 확대
- `J2_bm25_only`: BM25 단독 키워드 검색
- `J3_dense_bm25_rrf`: dense + BM25 결과를 RRF로 결합
- `J4_dense_rerank`: dense 후보를 reranker로 재정렬
- `J5_hybrid_rrf_rerank`: dense + BM25 + RRF 후보를 reranker로 재정렬

## 현재 설정되어 있는 중요 값 -> 원하시는 대로 수정하시면 됩니다!

```python
RUN_MODE = "exp100"
RUN_TAG = "0521"
CORPUS_OUTPUT_NAME = "parsing_p4_hwpx_125_0521"
PROJECT_ROOT = Path("/content/drive/MyDrive/DB_RAG_Codeit_Project")
EMBEDDING_MODEL_NAME = "nlpai-lab/KoE5"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
```

- `RUN_MODE`: 실행 규모 선택값입니다. 현재 `exp100`으로 설정되어 있습니다.
- `RUN_TAG`: 같은 조건의 이전 실험 결과와 섞이지 않도록 출력 폴더/Chroma 이름 뒤에 붙이는 실행 태그입니다.
- `CORPUS_OUTPUT_NAME`: 읽어올 corpus 산출물 폴더명입니다. 이번 버전은 `parsing_p4_hwpx_125_0521`입니다.
- `PROJECT_ROOT`: Google Drive 또는 GCP 작업 폴더의 프로젝트 루트 경로입니다. (필수 수정!!!)
- `EMBEDDING_MODEL_NAME`: dense 검색에 사용할 임베딩 모델입니다.
- `RERANKER_MODEL_NAME`: reranker 조건에서 사용할 cross-encoder 모델입니다.
- `FORCE_REBUILD`: 같은 collection을 재사용할지, Chroma collection을 새로 만들지 결정합니다. (처음에는 True)

## 결과 저장 위치

`exp100` 결과는 아래 폴더에 저장되도록 설정되어 있습니다. (원하시는 경로로 수정해 주세요!!)

```text
outputs/retrieval_quickcheck_p4_hwpx_125/exp100/
```

주요 파일은 요약 CSV, 실험별 results CSV, contexts JSONL, predictions JSONL입니다. 요약/지표처럼 표로 보는 결과는 CSV만 저장하고, 검색 context처럼 구조가 필요한 상세 근거만 JSONL로 저장합니다. 해석은 전체 지표와 단일문서/다중문서 분리 지표를 함께 보고 진행합니다.

## 주의

- Colab에서는 Chroma DB를 Google Drive가 아니라 `/content/...` 같은 런타임 로컬 경로에 만드는 것이 가장 안전합니다.
- GCP 또는 로컬에서는 `PROJECT_ROOT`만 실제 프로젝트 폴더로 바꾸면 됩니다.
- 이 노트북은 사용자가 생성한 `P4 HWPX 125` corpus를 직접 Chroma에 적재해서 확인하는 용도입니다.


In [1]:
# 0. 패키지/버전 충돌 방어
# Colab 버전 충돌 방지
# 본 셀은 실행 -> 런타임 1회 재시작 -> 다시 실행해 주세요!!!!!!!!!!!!!!!!
import importlib.metadata as importlib_metadata
import importlib.util
import re
import subprocess
import sys

PACKAGE_SPECS = [
    "chromadb",
    "sentence-transformers",
    "transformers>=4.56.0,<5",
    "tokenizers>=0.22.0,<0.23.1",
    "rank-bm25",
    "openpyxl",
    "opentelemetry-api",
    "opentelemetry-sdk",
    "opentelemetry-exporter-otlp-proto-grpc",
    "opentelemetry-exporter-otlp-proto-http",
    "opentelemetry-proto",
]
MODULE_TO_PIP = {
    "chromadb": "chromadb",
    "sentence_transformers": "sentence-transformers",
    "rank_bm25": "rank-bm25",
    "openpyxl": "openpyxl",
}

def package_version(package_name: str) -> str:
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return ""

def version_tuple(version_text: str) -> tuple[int, int, int]:
    nums = [int(x) for x in re.findall(r"\d+", version_text)[:3]]
    return tuple((nums + [0, 0, 0])[:3])

missing = [pip_name for module_name, pip_name in MODULE_TO_PIP.items() if importlib.util.find_spec(module_name) is None]
broken = {}

# tokenizers/transformers 버전 충돌 사전 감지
tok_version = package_version("tokenizers")
if tok_version and not ((0, 22, 0) <= version_tuple(tok_version) < (0, 23, 1)):
    broken["tokenizers"] = f"tokenizers=={tok_version}; expected >=0.22.0,<0.23.1"

for module_name in ["chromadb", "sentence_transformers"]:
    if importlib.util.find_spec(module_name) is None:
        continue
    try:
        __import__(module_name)
    except Exception as exc:
        broken[module_name] = repr(exc)

print("누락 패키지:", missing)
print("깨진 import:", broken)
print("설치 버전:", {
    "chromadb": package_version("chromadb"),
    "sentence-transformers": package_version("sentence-transformers"),
    "transformers": package_version("transformers"),
    "tokenizers": package_version("tokenizers"),
    "opentelemetry-api": package_version("opentelemetry-api"),
    "opentelemetry-sdk": package_version("opentelemetry-sdk"),
})

if missing or broken:
    print("검색 패키지 설치/업그레이드 중...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *PACKAGE_SPECS])
    print("설치 완료. import 오류가 계속되면 런타임을 재시작한 뒤 처음부터 다시 실행하세요.")
else:
    print("환경 준비 완료")


누락 패키지: []
깨진 import: {}
설치 버전: {'chromadb': '1.5.9', 'sentence-transformers': '5.5.1', 'transformers': '4.57.6', 'tokenizers': '0.22.2', 'opentelemetry-api': '1.42.0', 'opentelemetry-sdk': '1.42.0'}
환경 준비 완료


In [2]:
# Google Drive 연결
# Colab 환경 전용
# GCP/로컬 환경에서는 자동 생략
try:
    from google.colab import drive
except ModuleNotFoundError:
    print("Colab 환경이 아니므로 Google Drive 연결 생략")
else:
    drive.mount("/content/drive")
    print("Colab Google Drive 연결 완료")


Mounted at /content/drive
Colab Google Drive 연결 완료


In [3]:
# 1. 기본 설정
from __future__ import annotations

import ast
import hashlib
import json
import math
import os
import random
import re
import shutil
import time
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# 실행 모드 선택
# smoke: 연결 확인 / quick: 30문항 빠른 확인 / exp100: 100문항 6조건 비교 / full: 전체 평가
RUN_MODE = "exp100"
RUN_TAG = "0521"

# 실행 모드별 규모 설정
MODE_CONFIG = {
    "smoke": {"max_embed_chunks": 1000, "question_sample_size": 5, "force_rebuild": True, "experiment_suite": "dense_only"},
    "quick": {"max_embed_chunks": None, "question_sample_size": 30, "force_rebuild": True, "experiment_suite": "dense_only"},
    "exp100": {"max_embed_chunks": None, "question_sample_size": 100, "force_rebuild": True, "experiment_suite": "six_way"},
    "full": {"max_embed_chunks": None, "question_sample_size": None, "force_rebuild": True, "experiment_suite": "six_way"},
}
if RUN_MODE not in MODE_CONFIG:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE}. Use one of {list(MODE_CONFIG)}")

# 실행 결과 구분 태그
# 같은 RUN_MODE라도 날짜/수정 버전별 결과가 섞이지 않도록 출력 폴더와 Chroma 이름에 붙입니다.
RUN_OUTPUT_NAME = f"{RUN_MODE}_{RUN_TAG}" if RUN_TAG else RUN_MODE

# corpus 크기 설정
CORPUS_LIMIT = 125

# corpus 산출물 폴더명 설정
# 입력 corpus는 항상 새 0521 폴더를 사용합니다. RUN_TAG는 결과 저장 폴더 구분에만 사용합니다.
# 파일명에는 0521을 붙이지 않고, 폴더명으로만 새 버전을 분리합니다.
CORPUS_OUTPUT_NAME = "parsing_p4_hwpx_125_0521"

# 질문 샘플 고정값
RANDOM_SEED = 42

# 임베딩 모델 설정
# 비교 실험에서는 모델명을 고정해야 지표 비교가 가능합니다.
EMBEDDING_MODEL_NAME = "nlpai-lab/KoE5"

# 리랭커 모델 설정
# J4/J5 조건에서만 로드됩니다.
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"

# KoE5 입력 접두어 설정
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "

# 임베딩/Chroma 배치 크기 설정
EMBED_BATCH_SIZE = 64
CHROMA_ADD_BATCH_SIZE = 128
CHROMA_QUERY_BATCH_SIZE = 16

# 최종 문서 상위 k개 설정
DOC_TOP_K = 5

# 후보 생성 크기 설정
# top5 문서 평가 전 단계에서 dense/BM25/RRF/reranker가 볼 후보 수입니다.
BASE_DENSE_TOP_K = 40
WIDE_DENSE_TOP_K = 100
BM25_TOP_K = 100
RRF_TOP_K = 60
RERANK_KEEP_K = 40
RRF_K = 60
RERANK_BATCH_SIZE = 16
CANDIDATE_FAILURE_STRICT_TOP_K = 10
CANDIDATE_FAILURE_REFERENCE_TOP_K = 30

# Chroma 재생성 여부 설정
# True: 기존 collection 삭제 후 새로 적재
# False: 같은 corpus/model이면 기존 collection 재사용
FORCE_REBUILD = bool(MODE_CONFIG[RUN_MODE]["force_rebuild"])
# RUN_MODE 기반 실행 범위 적용
MAX_EMBED_CHUNKS = MODE_CONFIG[RUN_MODE]["max_embed_chunks"]
QUESTION_SAMPLE_SIZE = MODE_CONFIG[RUN_MODE]["question_sample_size"]
EXPERIMENT_SUITE = MODE_CONFIG[RUN_MODE]["experiment_suite"]

# 프로젝트 루트 경로 설정
# Colab 경로 예시: /content/drive/MyDrive/DB_RAG_Codeit_Project
# GCP/로컬 경로 예시: /home/.../project 또는 /Users/.../project
# 공유받은 사용자는 보통 이 값만 본인 환경에 맞게 수정
PROJECT_ROOT = Path("/content/drive/MyDrive/DB_RAG_Codeit_Project")

# corpus 산출물 경로 설정
PARSE_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / CORPUS_OUTPUT_NAME

# 실험 결과 저장 경로 설정
RUN_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / f'retrieval_quickcheck_p4_hwpx_{CORPUS_LIMIT}' / RUN_OUTPUT_NAME

# Colab/GCP 환경 구분
# Colab: Chroma DB는 Google Drive가 아니라 로컬 /content에 저장
# GCP/로컬: RUN_OUTPUT_DIR 아래 chroma_db에 저장
IS_COLAB = Path('/content').exists()
if IS_COLAB:
    CHROMA_DB_DIR = Path('/content') / f'chroma_p4_hwpx_{CORPUS_LIMIT}_{RUN_OUTPUT_NAME}'
else:
    CHROMA_DB_DIR = RUN_OUTPUT_DIR / 'chroma_db'

# 예측 결과 JSONL 저장 경로 설정
PREDICTION_DIR = RUN_OUTPUT_DIR / 'predictions'

# 결과 경로 안전장치
# 날짜 태그가 있는데 출력 폴더명에 태그가 없으면 이전 실험 결과와 섞일 수 있으므로 즉시 중단합니다.
if RUN_TAG and RUN_TAG not in RUN_OUTPUT_DIR.name:
    raise RuntimeError(f'RUN_TAG={RUN_TAG}가 결과 저장 경로에 반영되지 않았습니다: {RUN_OUTPUT_DIR}')

RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DB_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

# 입력 파일 경로 설정
CHUNKS_PATH = PARSE_OUTPUT_DIR / f'chunks_v2_{CORPUS_LIMIT}.jsonl'
PILOT_DOCS_PATH = PARSE_OUTPUT_DIR / f'pilot_docs_{CORPUS_LIMIT}.csv'
VALIDATION_REPORT_PATH = PARSE_OUTPUT_DIR / 'validation_report.json'
MANIFEST_PATH = PARSE_OUTPUT_DIR / 'manifest.json'
EVAL_DIR = PROJECT_ROOT / 'data' / 'eval'

# Chroma collection 이름 설정
# corpus 범위와 임베딩 모델이 바뀌면 collection 이름도 달라집니다.
model_digest = hashlib.sha1(EMBEDDING_MODEL_NAME.encode('utf-8')).hexdigest()[:8]
scope_token = f"{MAX_EMBED_CHUNKS}" if MAX_EMBED_CHUNKS else "all"
COLLECTION_NAME = f"p4hwpx{CORPUS_LIMIT}_{RUN_OUTPUT_NAME}_{scope_token}_{model_digest}"

print('실행 모드:', RUN_MODE)
print('실행 태그:', RUN_TAG)
print('실제 출력 폴더명:', RUN_OUTPUT_NAME)
print('실험 묶음:', EXPERIMENT_SUITE)
print('프로젝트 루트:', PROJECT_ROOT)
print('corpus 산출물 폴더명:', CORPUS_OUTPUT_NAME)
print('청크 파일 경로:', CHUNKS_PATH, '존재 여부=', CHUNKS_PATH.exists())
print('문서 목록 파일 경로:', PILOT_DOCS_PATH, '존재 여부=', PILOT_DOCS_PATH.exists())
print('manifest 파일 경로:', MANIFEST_PATH, '존재 여부=', MANIFEST_PATH.exists())
print('eval 폴더 경로:', EVAL_DIR, '존재 여부=', EVAL_DIR.exists())
print('결과 저장 경로:', RUN_OUTPUT_DIR)
print('Chroma DB 경로:', CHROMA_DB_DIR)
print('Chroma collection 이름:', COLLECTION_NAME)
print('Chroma 강제 재생성 여부:', FORCE_REBUILD)

실행 모드: exp100
실행 태그: 0521
실제 출력 폴더명: exp100_0521
실험 묶음: six_way
프로젝트 루트: /content/drive/MyDrive/DB_RAG_Codeit_Project
corpus 산출물 폴더명: parsing_p4_hwpx_125_0521
청크 파일 경로: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/parsing_p4_hwpx_125_0521/chunks_v2_125.jsonl 존재 여부= True
문서 목록 파일 경로: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/parsing_p4_hwpx_125_0521/pilot_docs_125.csv 존재 여부= True
manifest 파일 경로: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/parsing_p4_hwpx_125_0521/manifest.json 존재 여부= True
eval 폴더 경로: /content/drive/MyDrive/DB_RAG_Codeit_Project/data/eval 존재 여부= True
결과 저장 경로: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521
Chroma DB 경로: /content/chroma_p4_hwpx_125_exp100_0521
Chroma collection 이름: p4hwpx125_exp100_0521_all_388f1ca7
Chroma 강제 재생성 여부: True


In [4]:
# 2. 공통 함수
def normalize_doc_name(value: Any) -> str:
    text = unicodedata.normalize('NFC', str(value or '')).strip()
    text = Path(text).name
    while re.search(r'\.(hwp|hwpx|pdf|json)$', text, flags=re.I):
        text = re.sub(r'\.(hwp|hwpx|pdf|json)$', '', text, flags=re.I).strip()
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('인천공항운서비스㈜', '인천공항운영서비스㈜')
    return text.strip()

def parse_doc_list(value: Any) -> list[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    text = str(value).strip()
    if not text or text.lower() in {'nan', 'none', 'null'}:
        return []
    parsed = None
    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            break
        except Exception:
            pass
    if parsed is None:
        parsed = [text]
    if isinstance(parsed, str):
        parsed = [parsed]
    if not isinstance(parsed, (list, tuple, set)):
        return []
    docs = []
    for item in parsed:
        raw = item.get('filename') if isinstance(item, dict) else item
        norm = normalize_doc_name(raw)
        if norm:
            docs.append(norm)
    return docs

def scalar_metadata(value: Any) -> str | int | float | bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, int):
        return value
    if isinstance(value, float):
        return "" if math.isnan(value) else value
    if value is None:
        return ""
    if isinstance(value, (list, tuple, set)):
        return " | ".join(str(v) for v in value if str(v).strip())[:800]
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False, sort_keys=True)[:800]
    return str(value)[:1000]

def dcg(relevances: list[int]) -> float:
    return sum(rel / math.log2(idx + 2) for idx, rel in enumerate(relevances))

def doc_metrics(gt_docs: list[str], retrieved_docs: list[str], top_k: int = DOC_TOP_K) -> dict[str, float]:
    gt_set = {normalize_doc_name(doc) for doc in gt_docs if normalize_doc_name(doc)}
    pred = [normalize_doc_name(doc) for doc in retrieved_docs[:top_k]]
    if not gt_set:
        return {
            'hit_at_5': math.nan,
            'mrr_at_5': math.nan,
            'ndcg_at_5': math.nan,
            'doc_recall_at_5': math.nan,
            'multi_doc_recall_at_5': math.nan,
        }
    rel = [1 if doc in gt_set else 0 for doc in pred]
    first = next((idx + 1 for idx, v in enumerate(rel) if v), None)
    ideal = [1] * min(len(gt_set), top_k)
    recall = len(set(pred[:top_k]) & gt_set) / len(gt_set)
    return {
        'hit_at_5': float(any(rel)),
        'mrr_at_5': 0.0 if first is None else 1.0 / first,
        'ndcg_at_5': 0.0 if not ideal else dcg(rel) / dcg(ideal),
        'doc_recall_at_5': recall,
        'multi_doc_recall_at_5': recall,
    }

def unique_docs_from_items(items: list[dict], top_k: int = DOC_TOP_K) -> list[str]:
    docs = []
    seen = set()
    for item in items:
        doc = normalize_doc_name(item.get('norm_source_file') or item.get('source_file') or item.get('filename') or item.get('doc_id') or '')
        if not doc or doc in seen:
            continue
        seen.add(doc)
        docs.append(doc)
        if len(docs) >= top_k:
            break
    return docs

def unique_docs_from_query_result(metadatas: list[dict], top_k: int = DOC_TOP_K) -> list[str]:
    return unique_docs_from_items(metadatas, top_k)

def safe_short_text(text: str, n: int = 260) -> str:
    text = re.sub(r'\s+', ' ', str(text or '')).strip()
    return text[:n] + (' ...' if len(text) > n else '')

def looks_noisy_or_typo(text: Any) -> bool:
    # Eval의 E 타입처럼 의도적으로 깨진/구어체 질문을 대략 분리하기 위한 간단한 표시값입니다.
    value = str(text or '')
    typo_markers = ['쯤', '쫌', '머', '뭐임', '됴', '쑤', '츄', '쩡', '씨스', '시트', '뽀', '구츅', '확꾜']
    return any(marker in value for marker in typo_markers)

def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

def file_sha1(path: Path, chunk_size: int = 1024 * 1024) -> str:
    # manifest와 실제 청크 파일이 같은 버전인지 확인하기 위한 파일 hash 계산
    hasher = hashlib.sha1()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            hasher.update(chunk)
    return hasher.hexdigest()


In [5]:
# 3. P4 125 corpus 및 eval 질문 로드
if not CHUNKS_PATH.exists():
    raise FileNotFoundError(CHUNKS_PATH)
if not PILOT_DOCS_PATH.exists():
    raise FileNotFoundError(PILOT_DOCS_PATH)
if not EVAL_DIR.exists():
    raise FileNotFoundError(EVAL_DIR)
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(MANIFEST_PATH)

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
expected_chunks_sha1 = (manifest.get('file_hashes') or {}).get('chunks_v2_sha1', '')
actual_chunks_sha1 = file_sha1(CHUNKS_PATH)
print('manifest chunks_v2_sha1:', expected_chunks_sha1)
print('actual   chunks_v2_sha1:', actual_chunks_sha1)
if expected_chunks_sha1 and expected_chunks_sha1 != actual_chunks_sha1:
    raise RuntimeError('manifest.json의 chunks_v2 hash와 실제 chunks_v2_125.jsonl hash가 다릅니다. 서로 다른 버전 산출물이 섞였을 가능성이 큽니다.')

if VALIDATION_REPORT_PATH.exists():
    report = json.loads(VALIDATION_REPORT_PATH.read_text(encoding='utf-8'))
    print('corpus 검증:', {k: report.get(k) for k in ['status', 'document_count', 'eval_physical_source_docs_included', 'additional_sampled_docs', 'chunk_count', 'embed_enabled_count', 'chunks_jsonl_file_size_mib']})
    if report.get('chunks_jsonl_sha1') and report.get('chunks_jsonl_sha1') != actual_chunks_sha1:
        raise RuntimeError('validation_report.json의 chunks hash와 실제 chunks 파일 hash가 다릅니다.')

pilot_df = pd.read_csv(PILOT_DOCS_PATH)
pilot_norms = set(pilot_df['norm_name'].map(normalize_doc_name))
print('선택 문서 수:', len(pilot_df), '정규화 문서 수:', len(pilot_norms))

chunk_rows = []
with CHUNKS_PATH.open('r', encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        if not row.get('embed_enabled', False):
            continue
        source_file = row.get('source_file') or row.get('metadata', {}).get('source_file', '')
        meta = dict(row.get('metadata') or {})
        chunk_type = row.get('chunk_type') or meta.get('chunk_type', '')
        fact_type = row.get('fact_type') or meta.get('fact_type', '')
        section_path = row.get('section_path') or meta.get('section_path', '')
        table_role = row.get('table_role') or meta.get('table_role', '')
        meta.update({
            'chunk_id': row.get('chunk_id', ''),
            'doc_id': row.get('doc_id', ''),
            'source_file': source_file,
            'norm_source_file': normalize_doc_name(source_file),
            'chunk_type': chunk_type,
            'fact_type': fact_type,
            'section_path': section_path,
            'table_role': table_role,
        })
        clean_meta = {k: scalar_metadata(v) for k, v in meta.items() if scalar_metadata(v) != ''}
        chunk_rows.append({
            'chunk_id': row.get('chunk_id'),
            'doc_id': row.get('doc_id', ''),
            'source_file': source_file,
            'norm_source_file': normalize_doc_name(source_file),
            'chunk_type': chunk_type,
            'fact_type': fact_type,
            'section_path': section_path,
            'table_role': table_role,
            'content': row.get('content', ''),
            'metadata': clean_meta,
        })

chunks_df = pd.DataFrame(chunk_rows)
if MAX_EMBED_CHUNKS is not None:
    chunks_df = chunks_df.head(int(MAX_EMBED_CHUNKS)).copy()
chunks_df = chunks_df.reset_index(drop=True)
print('Chroma 적재 대상 청크 수:', len(chunks_df))
print('chunk_type 분포:', chunks_df['chunk_type'].value_counts().to_dict())
print('fact_type 분포:', chunks_df.loc[chunks_df['chunk_type'].eq('fact_candidates'), 'fact_type'].value_counts().to_dict())
print('선택 청크의 고유 문서 수:', chunks_df['norm_source_file'].nunique())
if chunks_df['chunk_id'].duplicated().any():
    raise ValueError('duplicate chunk_id in selected chunks')

eval_frames = []
for csv_path in sorted(EVAL_DIR.glob('*.csv')):
    df = pd.read_csv(csv_path)
    df['source_eval_file'] = csv_path.name
    eval_frames.append(df)
eval_df = pd.concat(eval_frames, ignore_index=True)
eval_df['ground_truth_doc_list'] = eval_df['ground_truth_docs'].apply(parse_doc_list)
eval_df['gt_norm_set'] = eval_df['ground_truth_doc_list'].apply(lambda docs: {normalize_doc_name(doc) for doc in docs})
available_norms = set(chunks_df['norm_source_file'])
eval_df['gt_all_in_pilot'] = eval_df['gt_norm_set'].apply(lambda docs: bool(docs) and docs.issubset(pilot_norms))
eval_df['gt_all_in_loaded_chunks'] = eval_df['gt_norm_set'].apply(lambda docs: bool(docs) and docs.issubset(available_norms))
eval_df['is_multi_doc'] = eval_df['gt_norm_set'].apply(lambda docs: len(docs) > 1)
eval_df['is_noisy_or_typo'] = eval_df['question'].apply(looks_noisy_or_typo)
eval_df['normalized_type'] = eval_df['type'].astype(str).str.strip().str.upper()
eval_df['normalized_difficulty'] = eval_df['difficulty'].astype(str).str.strip()
eligible_df = eval_df[eval_df['gt_all_in_pilot'] & eval_df['gt_all_in_loaded_chunks']].copy().reset_index(drop=True)
print('eval 전체 질문 수:', len(eval_df), '현재 corpus로 평가 가능한 질문 수:', len(eligible_df))
print('평가 가능 질문 유형 분포:', eligible_df['normalized_type'].value_counts().to_dict())
print('평가 가능 다중문서 질문 수:', int(eligible_df['is_multi_doc'].sum()))
print('평가 가능 난독화/오타 질문 수:', int(eligible_df['is_noisy_or_typo'].sum()))
if eligible_df.empty:
    raise RuntimeError('No eligible eval questions for the currently loaded chunk scope. Use RUN_MODE="quick" or increase max_embed_chunks.')

manifest chunks_v2_sha1: 1d7ad9a0105051241ae4dc60335aedff8ceebe49
actual   chunks_v2_sha1: 1d7ad9a0105051241ae4dc60335aedff8ceebe49
corpus 검증: {'status': 'PASS', 'document_count': 125, 'eval_physical_source_docs_included': 40, 'additional_sampled_docs': 85, 'chunk_count': 22408, 'embed_enabled_count': 18295, 'chunks_jsonl_file_size_mib': 56.16}
선택 문서 수: 125 정규화 문서 수: 125
Chroma 적재 대상 청크 수: 18295
chunk_type 분포: {'table': 11541, 'text': 5795, 'fact_candidates': 959}
fact_type 분포: {'document_summary': 216, 'business_type': 125, 'submission_logistics': 118, 'duration': 118, 'eligibility': 118, 'submission_documents': 113, 'budget': 102, 'bid_deadline': 49}
선택 청크의 고유 문서 수: 125
eval 전체 질문 수: 500 현재 corpus로 평가 가능한 질문 수: 500
평가 가능 질문 유형 분포: {'B': 200, 'A': 150, 'C': 50, 'D': 50, 'E': 50}
평가 가능 다중문서 질문 수: 203
평가 가능 난독화/오타 질문 수: 33


In [6]:
# 4. RUN_MODE별 질문 선택
def sample_diverse_questions(df: pd.DataFrame, n: int | None, seed: int = RANDOM_SEED) -> pd.DataFrame:
    if n is None or len(df) <= n:
        return df.copy().reset_index(drop=True)
    rng = random.Random(seed)
    work = df.copy()
    work['stratum'] = work['normalized_type'].astype(str) + '|' + work['normalized_difficulty'].astype(str) + '|multi=' + work['is_multi_doc'].astype(str)
    selected = []
    for _, group in work.groupby('stratum', sort=True):
        selected.append(rng.choice(group.index.tolist()))
        if len(selected) >= n:
            break
    remaining = [idx for idx in work.index.tolist() if idx not in set(selected)]
    rng.shuffle(remaining)
    selected.extend(remaining[: max(0, n - len(selected))])
    sampled = work.loc[selected[:n]].copy()
    return sampled.sort_values(['is_multi_doc', 'normalized_type', 'id'], ascending=[False, True, True]).reset_index(drop=True)

questions_df = sample_diverse_questions(eligible_df, QUESTION_SAMPLE_SIZE, RANDOM_SEED)
print('선택 질문 수:', len(questions_df))
print('질문 유형 분포:', questions_df['normalized_type'].value_counts().to_dict())
print('난이도 분포:', questions_df['normalized_difficulty'].value_counts().to_dict())
print('다중문서 질문 수:', int(questions_df['is_multi_doc'].sum()))
display(questions_df[['id', 'normalized_type', 'normalized_difficulty', 'is_multi_doc', 'question', 'ground_truth_docs']].head(5))


선택 질문 수: 100
질문 유형 분포: {'B': 35, 'A': 31, 'E': 14, 'D': 11, 'C': 9}
난이도 분포: {'하': 45, '중': 34, '상': 21}
다중문서 질문 수: 36


,id,normalized_type,normalized_difficulty,is_multi_doc,question,ground_truth_docs
0,Q007,B,하,True,한국가스공사의 '차세대 ERP 구축' 사업과 고려대학교의 '차세대 포털·학사 정보시...,"[""한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp"", ""고려대학교..."
1,Q008,B,하,True,한국수자원공사의 '용인 첨단 시스템반도체 국가산단 용수공급사업'과 한국수출입은행의 ...,"[""한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp"", ""..."
2,Q012,B,중,True,나노종합기술원의 스마트 팹 서비스 관련 설비온라인 사업과 부산국제영화제의 BIFF ...,"[""나노종합기술원_스마트 팹 서비스 활용체계 구축관련 설비온라인 시스.hwp"", ""..."
3,Q049,B,하,True,나노종합기술원의 '스마트 팹 서비스 설비온라인 사업'과 인천광역시의 '도시계획위원회...,"[""나노종합기술원_스마트 팹 서비스 활용체계 구축관련 설비온라인 시스.hwp"", ""..."
4,Q054,B,상,True,"한국가스공사의 '차세대 통합정보시스템', 나노종합기술원의 '설비온라인 시스템', 그...","[""한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp"", ""나노종합기..."


In [7]:
# 5. Chroma collection 생성/재사용
import chromadb
from chromadb.config import Settings
import torch
from sentence_transformers import SentenceTransformer

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm.auto import tqdm

print('chromadb:', chromadb.__version__)
print('torch:', torch.__version__)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('실행 장치:', device)
print('임베딩 모델:', EMBEDDING_MODEL_NAME)

client = chromadb.PersistentClient(
    path=str(CHROMA_DB_DIR),
    settings=Settings(anonymized_telemetry=False),
)

expected_count = len(chunks_df)
corpus_signature_payload = {
    'corpus_output_name': CORPUS_OUTPUT_NAME,
    'chunks_path': str(CHUNKS_PATH),
    'chunks_size': CHUNKS_PATH.stat().st_size,
    'chunks_mtime': round(CHUNKS_PATH.stat().st_mtime, 6),
    'selected_chunk_count': int(expected_count),
    'first_chunk_id': str(chunks_df.iloc[0]['chunk_id']) if expected_count else '',
    'last_chunk_id': str(chunks_df.iloc[-1]['chunk_id']) if expected_count else '',
    'embedding_model': EMBEDDING_MODEL_NAME,
    'run_mode': RUN_MODE,
    'max_embed_chunks': MAX_EMBED_CHUNKS,
}
CORPUS_SIGNATURE = hashlib.sha1(json.dumps(corpus_signature_payload, sort_keys=True).encode('utf-8')).hexdigest()[:16]
existing_count = None
existing_signature = None
try:
    collection = client.get_collection(COLLECTION_NAME)
    existing_count = collection.count()
    existing_signature = (collection.metadata or {}).get('corpus_signature')
    print('기존 collection 청크 수:', existing_count)
    print('기존 corpus_signature:', existing_signature)
except Exception:
    collection = None
    print('기존 collection 없음')
print('현재 corpus_signature:', CORPUS_SIGNATURE)

if FORCE_REBUILD and collection is not None:
    print('Chroma 강제 재생성으로 기존 collection 삭제:', COLLECTION_NAME)
    client.delete_collection(COLLECTION_NAME)
    collection = None
    existing_count = None

if collection is not None and (existing_count != expected_count or existing_signature != CORPUS_SIGNATURE):
    print(
        '청크 수 또는 signature가 달라 기존 collection 삭제:',
        {'existing_count': existing_count, 'expected_count': expected_count, 'existing_signature': existing_signature, 'expected_signature': CORPUS_SIGNATURE},
    )
    client.delete_collection(COLLECTION_NAME)
    collection = None
    existing_count = None
    existing_signature = None

if collection is None:
    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={
            'hnsw:space': 'cosine',
            'corpus': CORPUS_OUTPUT_NAME,
            'run_mode': RUN_MODE,
            'embedding_model': EMBEDDING_MODEL_NAME,
            'corpus_signature': CORPUS_SIGNATURE,
            'selected_chunk_count': str(expected_count),
        },
    )
    should_add = True
else:
    should_add = existing_count != expected_count

model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)

if should_add:
    started_all = time.perf_counter()
    total_batches = math.ceil(expected_count / CHROMA_ADD_BATCH_SIZE)
    print(f'Chroma collection 생성: {expected_count}개 청크, {total_batches}개 배치')
    progress = tqdm(
        total=expected_count,
        desc=f'임베딩 + Chroma 적재 ({RUN_MODE})',
        unit='chunk',
        dynamic_ncols=True,
        leave=True,
    )
    progress.set_postfix(batch=f'0/{total_batches}', embed='대기', add='대기', count=collection.count(), refresh=True)
    progress.refresh()
    print('진행률 표시 시작; 임베딩 시작')
    try:
        for batch_no, start in enumerate(range(0, expected_count, CHROMA_ADD_BATCH_SIZE), start=1):
            end = min(start + CHROMA_ADD_BATCH_SIZE, expected_count)
            batch = chunks_df.iloc[start:end]
            ids = batch['chunk_id'].astype(str).tolist()
            documents = batch['content'].astype(str).tolist()
            metadatas = batch['metadata'].tolist()
            embed_started = time.perf_counter()
            embeddings = model.encode(
                [PASSAGE_PREFIX + doc for doc in documents],
                batch_size=EMBED_BATCH_SIZE,
                normalize_embeddings=True,
                show_progress_bar=False,
            ).astype('float32').tolist()
            embed_sec = time.perf_counter() - embed_started
            add_started = time.perf_counter()
            collection.add(ids=ids, documents=documents, metadatas=metadatas, embeddings=embeddings)
            add_sec = time.perf_counter() - add_started
            current_count = collection.count()
            progress.update(len(batch))
            progress.set_postfix(
                batch=f'{batch_no}/{total_batches}',
                embed=f'{embed_sec:.2f}s',
                add=f'{add_sec:.2f}s',
                count=current_count,
                refresh=True,
            )
    finally:
        progress.close()
    print('색인 소요 초:', round(time.perf_counter() - started_all, 2))
else:
    print('collection이 이미 완성되어 적재 생략')

print('collection 이름:', COLLECTION_NAME)
print('collection 청크 수:', collection.count())
if collection.count() != expected_count:
    raise RuntimeError(f'collection count mismatch: {collection.count()} != {expected_count}')

chromadb: 1.5.9
torch: 2.10.0+cu128
실행 장치: cuda
임베딩 모델: nlpai-lab/KoE5
기존 collection 없음
현재 corpus_signature: 40f3f9b514000f8a


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Chroma collection 생성: 18295개 청크, 143개 배치


임베딩 + Chroma 적재 (exp100):   0%|          | 0/18295 [00:00<?, ?chunk/s]

진행률 표시 시작; 임베딩 시작
색인 소요 초: 722.22
collection 이름: p4hwpx125_exp100_0521_all_388f1ca7
collection 청크 수: 18295


## 6개 retrieval 실험 조건 설명

이 셀은 `RUN_MODE="exp100"` 또는 `RUN_MODE="full"`일 때 6가지 검색 조건을 실제 코드로 만듭니다.

- `J0_dense_baseline`: 임베딩 벡터 유사도만 사용합니다. 의미가 비슷한 질문에는 강하지만 숫자/고유명사 누락이 생길 수 있습니다.
- `J1_dense_wide`: dense 후보 수를 늘립니다. 정답 문서가 후보 생성 단계에서 빠지는지 확인합니다.
- `J2_bm25_only`: BM25 키워드 검색만 사용합니다. 숫자, 기관명, 공고명처럼 표면 단어가 중요한 질문을 확인합니다.
- `J3_dense_bm25_rrf`: dense와 BM25 결과를 RRF로 합칩니다. 의미 검색과 키워드 검색을 함께 쓰되 reranker는 쓰지 않습니다.
- `J4_dense_rerank`: dense 후보를 reranker가 다시 읽고 재정렬합니다. 후보 생성은 dense에만 의존합니다.
- `J5_hybrid_rrf_rerank`: dense와 BM25를 RRF로 합친 뒤 reranker로 재정렬합니다. 품질 확인용으로 가장 강하지만 가장 느립니다.

해석 기준은 특정 지표 하나가 아니라 `hit_at_5`, `mrr_at_5`, `ndcg_at_5`, `doc_recall_at_5`, 단일/다중문서 분리 지표, 실행 시간을 함께 봅니다.


In [8]:
# 6. 검색 함수 및 실험 조건 정의
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder


# BM25 토큰화 함수
def tokenize(text: str) -> list[str]:
    return re.findall(r"[0-9A-Za-z가-힣]+", str(text).lower())

# BM25/reranker 필요 시점 로딩 변수
bm25 = None
bm25_tokens = None
reranker = None

# BM25 색인 생성
def get_bm25():
    global bm25, bm25_tokens
    if bm25 is None:
        bm25_tokens = [tokenize(text) for text in chunks_df['content'].astype(str).tolist()]
        bm25 = BM25Okapi(bm25_tokens)
        print('BM25 문서 수:', len(bm25_tokens))
    return bm25

# reranker 모델 로드
def get_reranker():
    global reranker
    if reranker is None:
        print('리랭커 모델 로드:', RERANKER_MODEL_NAME, 'device:', device)
        reranker = CrossEncoder(RERANKER_MODEL_NAME, device=device)
    return reranker

def result_from_chunk_row(row: pd.Series, rank: int, score: float, method: str) -> dict:
    meta = dict(row.get('metadata') or {})
    return {
        'rank': int(rank),
        'score': float(score),
        'method': method,
        'chunk_id': str(row.get('chunk_id', '')),
        'doc_id': str(row.get('doc_id', '')),
        'source_file': str(row.get('source_file', '')),
        'norm_source_file': normalize_doc_name(row.get('norm_source_file') or row.get('source_file') or ''),
        'filename': str(row.get('source_file', '')),
        'chunk_type': str(row.get('chunk_type', '')),
        'fact_type': str(row.get('fact_type', '')),
        'section_path': str(row.get('section_path', '')),
        'table_role': str(row.get('table_role', '')),
        'text': str(row.get('content', '')),
        'metadata': meta,
    }

# Dense 검색
# 질문을 KoE5 질문 임베딩으로 바꾼 뒤 Chroma에서 가까운 청크를 찾습니다.
def dense_search(query: str, top_k: int) -> list[dict]:
    query_embedding = model.encode([QUERY_PREFIX + query], normalize_embeddings=True).astype('float32').tolist()
    result = collection.query(query_embeddings=query_embedding, n_results=top_k, include=['documents', 'metadatas', 'distances'])
    ids = result.get('ids', [[]])[0]
    docs = result.get('documents', [[]])[0]
    metas = result.get('metadatas', [[]])[0]
    distances = result.get('distances', [[]])[0]
    rows = []
    for idx, chunk_id in enumerate(ids):
        meta = metas[idx] if idx < len(metas) else {}
        rows.append({
            'rank': idx + 1,
            'score': -float(distances[idx]) if idx < len(distances) else 0.0,
            'distance': float(distances[idx]) if idx < len(distances) else math.nan,
            'method': 'dense',
            'chunk_id': str(chunk_id),
            'doc_id': str(meta.get('doc_id') or ''),
            'source_file': str(meta.get('source_file') or meta.get('filename') or ''),
            'norm_source_file': normalize_doc_name(meta.get('norm_source_file') or meta.get('source_file') or ''),
            'filename': str(meta.get('source_file') or meta.get('filename') or ''),
            'chunk_type': str(meta.get('chunk_type') or ''),
            'fact_type': str(meta.get('fact_type') or ''),
            'section_path': str(meta.get('section_path') or ''),
            'table_role': str(meta.get('table_role') or ''),
            'text': docs[idx] if idx < len(docs) else '',
            'metadata': meta,
        })
    return rows

# BM25 검색
# 단어 겹침이 강한 청크를 찾습니다. 숫자/기관명/고유명사 질문에서 보완 효과가 있습니다.
def bm25_search(query: str, top_k: int) -> list[dict]:
    scores = get_bm25().get_scores(tokenize(query))
    if len(scores) == 0:
        return []
    top_idx = np.argsort(scores)[::-1][:top_k]
    rows = []
    for rank, idx in enumerate(top_idx, start=1):
        row = chunks_df.iloc[int(idx)]
        rows.append(result_from_chunk_row(row, rank, float(scores[int(idx)]), 'bm25'))
    return rows

# RRF 결합
# dense와 BM25의 순위를 점수로 합쳐 한쪽 검색 방식의 누락을 줄입니다.
def rrf_fuse(result_lists: list[list[dict]], top_k: int, rrf_k: int = RRF_K) -> list[dict]:
    fused = {}
    first_seen = {}
    for result_list in result_lists:
        for item in result_list:
            cid = item['chunk_id']
            fused[cid] = fused.get(cid, 0.0) + 1.0 / (rrf_k + int(item.get('rank', 9999)))
            first_seen.setdefault(cid, item)
    ordered = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:top_k]
    output = []
    for rank, (cid, score) in enumerate(ordered, start=1):
        item = dict(first_seen[cid])
        item['rank'] = rank
        item['score'] = float(score)
        item['method'] = 'rrf'
        output.append(item)
    return output

# 리랭킹
# 후보 청크를 cross-encoder가 다시 읽고 질문과 더 맞는 순서로 재정렬합니다.
def rerank(query: str, candidates: list[dict], keep_k: int = RERANK_KEEP_K) -> list[dict]:
    if not candidates:
        return []
    pairs = [(query, item['text']) for item in candidates]
    scores = get_reranker().predict(pairs, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False)
    ranked = []
    for item, score in zip(candidates, scores):
        new_item = dict(item)
        new_item['rerank_score'] = float(score)
        ranked.append(new_item)
    ranked.sort(key=lambda x: x['rerank_score'], reverse=True)
    for rank, item in enumerate(ranked[:keep_k], start=1):
        item['rank'] = rank
        item['method'] = item.get('method', '') + '_reranked'
    return ranked[:keep_k]

# 실험 조건 객체
@dataclass
class RetrievalExperiment:
    experiment_id: str
    method: str
    dense_top_k: int = BASE_DENSE_TOP_K
    bm25_top_k: int = BM25_TOP_K
    rrf_top_k: int = RRF_TOP_K
    rerank_keep_k: int = RERANK_KEEP_K
    final_doc_top_k: int = DOC_TOP_K

# 6가지 실험 조건 설정
# J0: dense 단독 기준선
# J1: dense 후보 수 확대
# J2: BM25 단독
# J3: dense + BM25 RRF 결합
# J4: dense 후보 리랭킹
# J5: dense + BM25 RRF 후 리랭킹
if EXPERIMENT_SUITE == 'six_way':
    experiments = [
        RetrievalExperiment('J0_dense_baseline', 'dense', dense_top_k=BASE_DENSE_TOP_K),
        RetrievalExperiment('J1_dense_wide', 'dense', dense_top_k=WIDE_DENSE_TOP_K),
        RetrievalExperiment('J2_bm25_only', 'bm25'),
        RetrievalExperiment('J3_dense_bm25_rrf', 'hybrid_rrf'),
        RetrievalExperiment('J4_dense_rerank', 'dense_rerank'),
        RetrievalExperiment('J5_hybrid_rrf_rerank', 'hybrid_rrf_rerank'),
    ]
else:
    # smoke/quick 기본 조건
    experiments = [RetrievalExperiment('J0_dense_baseline', 'dense', dense_top_k=BASE_DENSE_TOP_K)]

print('실험 조건:', [exp.experiment_id for exp in experiments])

# 실험별 검색 실행 함수
def run_retrieval(query: str, exp: RetrievalExperiment) -> tuple[list[dict], list[dict], dict]:
    started = time.perf_counter()
    dense_ms = bm25_ms = rrf_ms = rerank_ms = 0.0
    dense_results = []
    bm25_results = []
    candidate_results = []

    if exp.method == 'dense':
        dense_started = time.perf_counter()
        dense_results = dense_search(query, exp.dense_top_k)
        dense_ms = (time.perf_counter() - dense_started) * 1000
        candidate_results = dense_results
        final_results = dense_results
    elif exp.method == 'bm25':
        bm25_started = time.perf_counter()
        bm25_results = bm25_search(query, exp.bm25_top_k)
        bm25_ms = (time.perf_counter() - bm25_started) * 1000
        candidate_results = bm25_results
        final_results = bm25_results
    elif exp.method == 'hybrid_rrf':
        dense_started = time.perf_counter()
        dense_results = dense_search(query, exp.dense_top_k)
        dense_ms = (time.perf_counter() - dense_started) * 1000
        bm25_started = time.perf_counter()
        bm25_results = bm25_search(query, exp.bm25_top_k)
        bm25_ms = (time.perf_counter() - bm25_started) * 1000
        rrf_started = time.perf_counter()
        candidate_results = rrf_fuse([dense_results, bm25_results], exp.rrf_top_k)
        rrf_ms = (time.perf_counter() - rrf_started) * 1000
        final_results = candidate_results
    elif exp.method == 'dense_rerank':
        dense_started = time.perf_counter()
        dense_results = dense_search(query, exp.dense_top_k)
        dense_ms = (time.perf_counter() - dense_started) * 1000
        candidate_results = dense_results
        rerank_started = time.perf_counter()
        final_results = rerank(query, candidate_results, exp.rerank_keep_k)
        rerank_ms = (time.perf_counter() - rerank_started) * 1000
    elif exp.method == 'hybrid_rrf_rerank':
        dense_started = time.perf_counter()
        dense_results = dense_search(query, exp.dense_top_k)
        dense_ms = (time.perf_counter() - dense_started) * 1000
        bm25_started = time.perf_counter()
        bm25_results = bm25_search(query, exp.bm25_top_k)
        bm25_ms = (time.perf_counter() - bm25_started) * 1000
        rrf_started = time.perf_counter()
        candidate_results = rrf_fuse([dense_results, bm25_results], exp.rrf_top_k)
        rrf_ms = (time.perf_counter() - rrf_started) * 1000
        rerank_started = time.perf_counter()
        final_results = rerank(query, candidate_results, exp.rerank_keep_k)
        rerank_ms = (time.perf_counter() - rerank_started) * 1000
    else:
        raise ValueError(exp.method)

    total_ms = (time.perf_counter() - started) * 1000
    timings = {
        'retrieval_ms': total_ms,
        'dense_ms': dense_ms,
        'bm25_ms': bm25_ms,
        'rrf_ms': rrf_ms,
        'rerank_ms': rerank_ms,
    }
    return final_results, candidate_results, timings

실험 조건: ['J0_dense_baseline', 'J1_dense_wide', 'J2_bm25_only', 'J3_dense_bm25_rrf', 'J4_dense_rerank', 'J5_hybrid_rrf_rerank']


In [9]:
# 7. 검색 실험 실행 및 상세 결과 저장
try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm.auto import tqdm

# 전체 실험 결과 누적 리스트
all_result_rows = []
all_context_rows = []
prediction_paths = {}
started_suite = time.perf_counter()

# 실험 조건별 반복 실행
for exp in experiments:
    exp_rows = []
    exp_context_rows = []
    prediction_rows = []
    for _, qrow in tqdm(questions_df.iterrows(), total=len(questions_df), desc=exp.experiment_id):
        query = str(qrow['question'])
        final_results, candidate_results, timings = run_retrieval(query, exp)
        gt_docs = sorted(qrow['gt_norm_set'])
        retrieved_docs5 = unique_docs_from_items(final_results, exp.final_doc_top_k)
        # 후보 생성 실패 확인용 문서 목록
        candidate_docs10 = unique_docs_from_items(candidate_results, CANDIDATE_FAILURE_STRICT_TOP_K)
        candidate_docs30 = unique_docs_from_items(candidate_results, CANDIDATE_FAILURE_REFERENCE_TOP_K)
        metrics = doc_metrics(gt_docs, retrieved_docs5, exp.final_doc_top_k)
        first_rank = 0.0 if metrics['mrr_at_5'] == 0 else 1.0 / metrics['mrr_at_5']
        row = {
            'experiment_id': exp.experiment_id,
            'method': exp.method,
            'id': qrow.get('id'),
            'type': qrow.get('normalized_type'),
            'difficulty': qrow.get('normalized_difficulty'),
            'is_multi_doc': bool(qrow.get('is_multi_doc')),
            'is_noisy_or_typo': bool(qrow.get('is_noisy_or_typo')),
            'question': qrow.get('question'),
            'ground_truth_docs': json.dumps(gt_docs, ensure_ascii=False),
            'retrieved_docs_top5': json.dumps(retrieved_docs5, ensure_ascii=False),
            'candidate_docs_top10': json.dumps(candidate_docs10, ensure_ascii=False),
            'candidate_docs_top30': json.dumps(candidate_docs30, ensure_ascii=False),
            'candidate_generation_failed_top10': int(len(set(candidate_docs10) & set(gt_docs)) == 0),
            'candidate_generation_failed_top30': int(len(set(candidate_docs30) & set(gt_docs)) == 0),
            'partial_multi_doc_loss': int(bool(qrow.get('is_multi_doc')) and 0 < metrics['multi_doc_recall_at_5'] < 1),
            'low_rank_correct': int(metrics['hit_at_5'] == 1 and (first_rank >= 3 or metrics['ndcg_at_5'] < 0.75)),
            **metrics,
            **timings,
        }
        exp_rows.append(row)
        prediction_contexts = []
        for rank, item in enumerate(final_results, start=1):
            context = {
                'rank': int(rank),
                'text': item.get('text', ''),
                'filename': item.get('source_file') or item.get('filename') or '',
                'doc_id': item.get('norm_source_file') or item.get('doc_id') or '',
                'chunk_id': item.get('chunk_id', ''),
                'score': item.get('score', 0.0),
                'rerank_score': item.get('rerank_score', None),
                'metadata': item.get('metadata', {}),
            }
            prediction_contexts.append(context)
            exp_context_rows.append({
                'experiment_id': exp.experiment_id,
                'question_id': qrow.get('id'),
                'rank': int(rank),
                'score': float(item.get('score', 0.0)),
                'rerank_score': item.get('rerank_score', None),
                'method': item.get('method', ''),
                'source_file': item.get('source_file') or item.get('filename') or '',
                'norm_source_file': item.get('norm_source_file') or normalize_doc_name(item.get('source_file') or item.get('filename') or ''),
                'chunk_id': item.get('chunk_id', ''),
                'chunk_type': item.get('chunk_type', ''),
                'fact_type': item.get('fact_type', ''),
                'section_path': item.get('section_path', ''),
                'table_role': item.get('table_role', ''),
                'text': safe_short_text(item.get('text', ''), 500),
            })
        prediction_rows.append({
            'id': qrow.get('id'),
            'question': qrow.get('question'),
            'answer': '',
            'retrieved_contexts': prediction_contexts,
            'ground_truth_answer': qrow.get('ground_truth_answer', ''),
            'ground_truth_docs': qrow.get('ground_truth_docs', ''),
            'latency_ms': timings['retrieval_ms'],
            'retrieval_ms': timings['retrieval_ms'],
            'rerank_ms': timings['rerank_ms'],
            'model_name': 'retrieval_only',
            'embedding_model': EMBEDDING_MODEL_NAME,
            'retriever_config': {
                'corpus_name': f'p4_hwpx_{CORPUS_LIMIT}',
                'corpus_version': 'v2_hwpx_table_aware',
                'source_jsonl': str(CHUNKS_PATH),
                'embedding_model': EMBEDDING_MODEL_NAME,
                'reranker_model': RERANKER_MODEL_NAME if 'rerank' in exp.method else '',
                'vector_db': 'chroma',
                'retrieval_method': exp.method,
                'dense_top_k': exp.dense_top_k if 'dense' in exp.method or 'hybrid' in exp.method else 0,
                'bm25_top_k': exp.bm25_top_k if 'bm25' in exp.method or 'hybrid' in exp.method else 0,
                'rrf_top_k': exp.rrf_top_k if 'hybrid' in exp.method else 0,
                'rerank_keep_k': exp.rerank_keep_k if 'rerank' in exp.method else 0,
                'final_doc_top_k': exp.final_doc_top_k,
                'eval_sample_size': QUESTION_SAMPLE_SIZE,
            },
        })
    exp_results_df = pd.DataFrame(exp_rows)
    exp_contexts_df = pd.DataFrame(exp_context_rows)
    exp_results_path = RUN_OUTPUT_DIR / f'{exp.experiment_id}_results.csv'
    exp_contexts_path = RUN_OUTPUT_DIR / f'{exp.experiment_id}_contexts.jsonl'
    exp_predictions_path = PREDICTION_DIR / f'{exp.experiment_id}_predictions.jsonl'
    exp_results_df.to_csv(exp_results_path, index=False, encoding='utf-8-sig')
    write_jsonl(exp_contexts_path, exp_context_rows)
    write_jsonl(exp_predictions_path, prediction_rows)
    prediction_paths[exp.experiment_id] = exp_predictions_path
    all_result_rows.extend(exp_rows)
    all_context_rows.extend(exp_context_rows)
    print('저장 완료:', exp_results_path)
    print('저장 완료:', exp_contexts_path)
    print('저장 완료:', exp_predictions_path)

all_results_df = pd.DataFrame(all_result_rows)
all_contexts_df = pd.DataFrame(all_context_rows)
# 호환용 별칭 설정
# 빠른 확인용으로 첫 번째 실험 결과를 results_df/contexts_df로 둡니다.
results_df = all_results_df[all_results_df['experiment_id'].eq(experiments[0].experiment_id)].copy().reset_index(drop=True)
contexts_df = all_contexts_df[all_contexts_df['experiment_id'].eq(experiments[0].experiment_id)].copy().reset_index(drop=True)
print('전체 실험 소요 초:', round(time.perf_counter() - started_suite, 2))
print('전체 결과 표 크기:', all_results_df.shape)
print('전체 context 표 크기:', all_contexts_df.shape)

J0_dense_baseline:   0%|          | 0/100 [00:00<?, ?it/s]

저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/J0_dense_baseline_results.csv
저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/J0_dense_baseline_contexts.jsonl
저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/predictions/J0_dense_baseline_predictions.jsonl


J1_dense_wide:   0%|          | 0/100 [00:00<?, ?it/s]

저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/J1_dense_wide_results.csv
저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/J1_dense_wide_contexts.jsonl
저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/predictions/J1_dense_wide_predictions.jsonl


J2_bm25_only:   0%|          | 0/100 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/J4_dense_rerank_results.csv
저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/J4_dense_rerank_contexts.jsonl
저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/predictions/J4_dense_rerank_predictions.jsonl


J5_hybrid_rrf_rerank:   0%|          | 0/100 [00:00<?, ?it/s]

저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/J5_hybrid_rrf_rerank_results.csv
저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/J5_hybrid_rrf_rerank_contexts.jsonl
저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/predictions/J5_hybrid_rrf_rerank_predictions.jsonl
전체 실험 소요 초: 495.44
전체 결과 표 크기: (600, 26)
전체 context 표 크기: (38000, 14)


In [10]:
# 8. 지표 요약 및 기준선 대비 변화량
def safe_mean(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors='coerce')
    return float(values.mean()) if values.notna().any() else math.nan

def safe_quantile(series: pd.Series, q: float) -> float:
    values = pd.to_numeric(series, errors='coerce')
    return float(values.quantile(q)) if values.notna().any() else math.nan

def summarize_experiment(group: pd.DataFrame) -> dict[str, Any]:
    single = group[~group['is_multi_doc']].copy()
    multi = group[group['is_multi_doc']].copy()
    return {
        'experiment_id': group['experiment_id'].iloc[0],
        'method': group['method'].iloc[0],
        'num_eval_questions': int(len(group)),
        'single_doc_questions': int(len(single)),
        'multi_doc_questions': int(len(multi)),
        'hit_at_5_any': safe_mean(group['hit_at_5']),
        'mrr_at_5': safe_mean(group['mrr_at_5']),
        'ndcg_at_5': safe_mean(group['ndcg_at_5']),
        'doc_recall_at_5': safe_mean(group['doc_recall_at_5']),
        'single_doc_hit_at_5': safe_mean(single['hit_at_5']),
        'single_doc_mrr_at_5': safe_mean(single['mrr_at_5']),
        'single_doc_ndcg_at_5': safe_mean(single['ndcg_at_5']),
        'multi_doc_hit_at_5': safe_mean(multi['hit_at_5']),
        'multi_doc_mrr_at_5': safe_mean(multi['mrr_at_5']),
        'multi_doc_ndcg_at_5': safe_mean(multi['ndcg_at_5']),
        'multi_doc_recall_at_5': safe_mean(multi['multi_doc_recall_at_5']),
        # 후보 생성 실패 지표
        # top10은 엄격한 기준, top30은 참고 기준
        'candidate_generation_failed_top10': safe_mean(group['candidate_generation_failed_top10']),
        'candidate_generation_failed_top30': safe_mean(group['candidate_generation_failed_top30']),
        'partial_multi_doc_loss': safe_mean(group['partial_multi_doc_loss']),
        'miss_count': int(pd.to_numeric(group['hit_at_5'], errors='coerce').fillna(0).eq(0).sum()),
        'candidate_failed_top10_count': int(pd.to_numeric(group['candidate_generation_failed_top10'], errors='coerce').fillna(0).eq(1).sum()),
        'candidate_failed_top30_count': int(pd.to_numeric(group['candidate_generation_failed_top30'], errors='coerce').fillna(0).eq(1).sum()),
        'partial_multi_doc_loss_count': int(pd.to_numeric(group['partial_multi_doc_loss'], errors='coerce').fillna(0).eq(1).sum()),
        'low_rank_correct_count': int(pd.to_numeric(group['low_rank_correct'], errors='coerce').fillna(0).eq(1).sum()),
        'retrieval_ms_avg': safe_mean(group['retrieval_ms']),
        'retrieval_ms_p95': safe_quantile(group['retrieval_ms'], 0.95),
        'dense_ms_avg': safe_mean(group['dense_ms']),
        'bm25_ms_avg': safe_mean(group['bm25_ms']),
        'rrf_ms_avg': safe_mean(group['rrf_ms']),
        'rerank_ms_avg': safe_mean(group['rerank_ms']),
    }

# 실험별 요약 지표 생성
summary_df = pd.DataFrame([summarize_experiment(g) for _, g in all_results_df.groupby('experiment_id', sort=False)])
# J0 dense 기준선 대비 변화량 계산
baseline = summary_df[summary_df['experiment_id'].eq('J0_dense_baseline')].iloc[0].to_dict()
for col in [
    'hit_at_5_any',
    'mrr_at_5',
    'ndcg_at_5',
    'doc_recall_at_5',
    'single_doc_mrr_at_5',
    'multi_doc_ndcg_at_5',
    'multi_doc_recall_at_5',
    'candidate_generation_failed_top10',
    'candidate_generation_failed_top30',
    'partial_multi_doc_loss',
    'retrieval_ms_avg',
]:
    summary_df[f'delta_vs_J0_{col}'] = summary_df[col] - float(baseline[col])

# 요약/상세 결과 저장 경로
summary_csv_path = RUN_OUTPUT_DIR / f'experiment_summary_{RUN_OUTPUT_NAME}.csv'
all_results_path = RUN_OUTPUT_DIR / f'all_experiment_results_{RUN_OUTPUT_NAME}.csv'
all_contexts_path = RUN_OUTPUT_DIR / f'all_experiment_contexts_{RUN_OUTPUT_NAME}.jsonl'
summary_df.to_csv(summary_csv_path, index=False, encoding='utf-8-sig')
all_results_df.to_csv(all_results_path, index=False, encoding='utf-8-sig')
write_jsonl(all_contexts_path, all_context_rows)

print('저장 완료:', summary_csv_path)
print('저장 완료:', all_results_path)
print('저장 완료:', all_contexts_path)
print('candidate_generation_failed_top10: 엄격한 후보 생성 실패 기준; top30은 참고 기준')
print('partial_multi_doc_loss: 다중문서 질문에서 top5에 정답 문서 일부만 들어온 경우')

summary_display_cols = [
    'experiment_id', 'method', 'num_eval_questions', 'single_doc_questions', 'multi_doc_questions',
    'hit_at_5_any', 'mrr_at_5', 'ndcg_at_5', 'doc_recall_at_5',
    'single_doc_mrr_at_5', 'multi_doc_ndcg_at_5', 'multi_doc_recall_at_5',
    'candidate_generation_failed_top10', 'candidate_generation_failed_top30', 'partial_multi_doc_loss',
    'miss_count', 'candidate_failed_top10_count', 'partial_multi_doc_loss_count',
    'retrieval_ms_avg', 'retrieval_ms_p95', 'rerank_ms_avg',
]
display(
    summary_df[summary_display_cols].sort_values(
        ['hit_at_5_any', 'mrr_at_5', 'ndcg_at_5', 'doc_recall_at_5', 'candidate_generation_failed_top10'],
        ascending=[False, False, False, False, True],
    )
)

저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/experiment_summary_exp100_0521.csv
저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/all_experiment_results_exp100_0521.csv
저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/all_experiment_contexts_exp100_0521.jsonl
candidate_generation_failed_top10: 엄격한 후보 생성 실패 기준; top30은 참고 기준
partial_multi_doc_loss: 다중문서 질문에서 top5에 정답 문서 일부만 들어온 경우


,experiment_id,method,num_eval_questions,single_doc_questions,multi_doc_questions,hit_at_5_any,mrr_at_5,ndcg_at_5,doc_recall_at_5,single_doc_mrr_at_5,...,multi_doc_recall_at_5,candidate_generation_failed_top10,candidate_generation_failed_top30,partial_multi_doc_loss,miss_count,candidate_failed_top10_count,partial_multi_doc_loss_count,retrieval_ms_avg,retrieval_ms_p95,rerank_ms_avg
5,J5_hybrid_rrf_rerank,hybrid_rrf_rerank,100,64,36,1.00,0.942500,0.943688,0.979167,0.937500,...,0.942130,0.00,0.00,0.05,0,0,5,2585.410451,3480.832651,2403.319284
4,J4_dense_rerank,dense_rerank,100,64,36,0.99,0.950000,0.928564,0.953333,0.933594,...,0.898148,0.01,0.00,0.07,1,1,7,1767.176656,2188.003207,1733.833368
3,J3_dense_bm25_rrf,hybrid_rrf,100,64,36,0.99,0.908667,0.895994,0.946667,0.921875,...,0.851852,0.00,0.00,0.08,1,0,8,179.558213,297.636806,0.000000
0,J0_dense_baseline,dense,100,64,36,0.98,0.920833,0.899897,0.937500,0.917969,...,0.854167,0.01,0.00,0.09,2,1,9,29.845658,34.100748,0.000000
1,J1_dense_wide,dense,100,64,36,0.98,0.920833,0.899897,0.937500,0.917969,...,0.854167,0.01,0.00,0.09,2,1,9,39.293147,44.337382,0.000000
2,J2_bm25_only,bm25,100,64,36,0.93,0.803500,0.783194,0.855833,0.801823,...,0.738426,0.06,0.02,0.14,7,6,14,158.758683,270.652087,0.000000


## 실패 원인 상세 분석 설정

아래 셀은 특정 실험 결과를 골라 실패 사례를 자세히 보는 분석 셀입니다.

먼저 위의 summary 표에서 보고 싶은 실험 이름을 확인한 뒤, 아래 값을 바꿔 실행합니다.

```python
ANALYSIS_EXPERIMENT_ID = "J5_hybrid_rrf_rerank"
```

예를 들어 기준선을 보고 싶으면 `J0_dense_baseline`, 하이브리드+리랭커를 보고 싶으면 `J5_hybrid_rrf_rerank`로 설정합니다.

이 셀은 top-5 정답 없음, 다중문서 일부 누락, 정답은 있으나 순위가 낮은 케이스, 검색된 chunk_type/fact_type 분포, 질문 유형/난이도/난독화 여부별 지표를 함께 보여줍니다.


In [11]:
# 9. 실패 원인 상세 분석
# 요약표 확인 후 분석할 실험 이름 설정
# multi_doc_recall만 단독 기준으로 보지 않고 전체 지표와 함께 해석

# 실패 분석 기준 실험 설정
ANALYSIS_EXPERIMENT_ID = 'J5_hybrid_rrf_rerank'

print('분석 대상 실험:', ANALYSIS_EXPERIMENT_ID)
analysis_df = all_results_df[all_results_df['experiment_id'].eq(ANALYSIS_EXPERIMENT_ID)].copy().reset_index(drop=True)
analysis_contexts_df = all_contexts_df[all_contexts_df['experiment_id'].eq(ANALYSIS_EXPERIMENT_ID)].copy().reset_index(drop=True)

misses = analysis_df[pd.to_numeric(analysis_df['hit_at_5'], errors='coerce').fillna(0).eq(0)].copy()
partial = analysis_df[pd.to_numeric(analysis_df['partial_multi_doc_loss'], errors='coerce').fillna(0).eq(1)].copy()
low_rank = analysis_df[pd.to_numeric(analysis_df['low_rank_correct'], errors='coerce').fillna(0).eq(1)].copy()
print('Top-5 정답 없음 수:', len(misses), '/', len(analysis_df))
print('다중문서 일부 누락 수:', len(partial), '/', len(analysis_df))
print('정답은 있으나 낮은 순위 수:', len(low_rank), '/', len(analysis_df))

if len(misses):
    print('\nTop-5 정답 없음 사례:')
    display(misses[['id', 'type', 'difficulty', 'is_multi_doc', 'is_noisy_or_typo', 'question', 'ground_truth_docs', 'retrieved_docs_top5', 'candidate_generation_failed_top10', 'candidate_generation_failed_top30']].head(30))
else:
    print('\nTop-5 정답 없음 사례 없음')

if len(partial):
    print('\n다중문서 일부 누락 사례:')
    display(partial[['id', 'type', 'difficulty', 'question', 'ground_truth_docs', 'retrieved_docs_top5', 'multi_doc_recall_at_5']].head(30))
else:
    print('\n다중문서 일부 누락 없음')

if len(low_rank):
    print('\n정답은 있으나 순위가 낮은 사례:')
    display(low_rank[['id', 'type', 'difficulty', 'is_multi_doc', 'question', 'ground_truth_docs', 'retrieved_docs_top5', 'mrr_at_5', 'ndcg_at_5']].head(30))

print('\n질문 유형별 지표:')
display(analysis_df.groupby('type')[['hit_at_5', 'mrr_at_5', 'ndcg_at_5', 'doc_recall_at_5', 'multi_doc_recall_at_5', 'partial_multi_doc_loss']].mean().reset_index())
print('\n난이도별 지표:')
display(analysis_df.groupby('difficulty')[['hit_at_5', 'mrr_at_5', 'ndcg_at_5', 'doc_recall_at_5', 'multi_doc_recall_at_5', 'partial_multi_doc_loss']].mean().reset_index())
print('\n단일/다중문서별 지표:')
display(analysis_df.groupby('is_multi_doc')[['hit_at_5', 'mrr_at_5', 'ndcg_at_5', 'doc_recall_at_5', 'multi_doc_recall_at_5', 'partial_multi_doc_loss']].mean().reset_index())
print('\n난독화/오타 질문 여부별 지표:')
display(analysis_df.groupby('is_noisy_or_typo')[['hit_at_5', 'mrr_at_5', 'ndcg_at_5', 'doc_recall_at_5', 'multi_doc_recall_at_5', 'partial_multi_doc_loss']].mean().reset_index())

print('\n검색된 chunk_type 분포:')
display(analysis_contexts_df['chunk_type'].value_counts(dropna=False).rename_axis('chunk_type').reset_index(name='count'))
print('\n검색된 fact_type 분포:')
display(analysis_contexts_df[analysis_contexts_df['chunk_type'].eq('fact_candidates')]['fact_type'].value_counts(dropna=False).rename_axis('fact_type').reset_index(name='count'))

# 실패 태그 생성
# 다음 corpus/parser 수정 방향을 잡기 위한 간단한 분류
def tag_failure(row: pd.Series) -> str:
    q = str(row.get('question', ''))
    retrieved = ' '.join(parse_doc_list(row.get('retrieved_docs_top5', '[]')))
    gt = ' '.join(parse_doc_list(row.get('ground_truth_docs', '[]')))
    tags = []
    if bool(row.get('is_multi_doc')):
        tags.append('multi_doc')
    if row.get('partial_multi_doc_loss') == 1:
        tags.append('partial_loss')
    if re.search(r'예산|금액|사업비|원|억', q):
        tags.append('숫자/예산 질문')
    if re.search(r'표|평가|배점|제출|서류|자격', q):
        tags.append('표/요건 질문')
    if re.search(r'재공고|공고|기관|대학교|공사', q + retrieved + gt):
        tags.append('유사 문서/재공고/기관 alias')
    if looks_noisy_or_typo(q):
        tags.append('난독화/구어체')
    if bool(row.get('is_multi_doc')) and row.get('multi_doc_recall_at_5', 0) < 1:
        tags.append('한 문서 독점 가능')
    return ' | '.join(dict.fromkeys(tags)) or 'review_needed'

failure_focus_df = pd.concat([misses, partial, low_rank], ignore_index=True).drop_duplicates(['experiment_id', 'id'])
if len(failure_focus_df):
    failure_focus_df['failure_tags'] = failure_focus_df.apply(tag_failure, axis=1)
    failure_path = RUN_OUTPUT_DIR / f'failure_focus_{ANALYSIS_EXPERIMENT_ID}.csv'
    failure_focus_df.to_csv(failure_path, index=False, encoding='utf-8-sig')
    print('\n저장 완료:', failure_path)
    display(failure_focus_df[['id', 'type', 'difficulty', 'is_multi_doc', 'failure_tags', 'question', 'ground_truth_docs', 'retrieved_docs_top5']].head(50))
else:
    print('\n집중 분석할 실패 사례 없음')

분석 대상 실험: J5_hybrid_rrf_rerank
Top-5 정답 없음 수: 0 / 100
다중문서 일부 누락 수: 5 / 100
정답은 있으나 낮은 순위 수: 11 / 100

Top-5 정답 없음 사례 없음

다중문서 일부 누락 사례:


,id,type,difficulty,question,ground_truth_docs,retrieved_docs_top5,multi_doc_recall_at_5
4,Q054,B,상,"한국가스공사의 '차세대 통합정보시스템', 나노종합기술원의 '설비온라인 시스템', 그...","[""그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역"", ""나노종...","[""그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역"", ""한국가...",0.666667
31,Q433,B,상,"아시아물위원회사무국 우즈벡 관개 사업비, 부산관광공사 경영정보망 사업비, 광주문화재...","[""남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사"", ""부산...","[""부산관광공사_경영정보시스템 기능개선"", ""사단법인아시아물위원회사무국_우즈벡-키르...",0.750000
33,Q453,B,상,"코이카-아시아물위원회 우즈벡 관개망, 한국연구재단 실태조사 시스템(1.293억), ...","[""부산관광공사_경영정보시스템 기능개선"", ""사단법인아시아물위원회사무국_우즈벡-키르...","[""KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원...",0.500000
34,Q473,B,중,아시아물위원회사무국 우즈베키스탄 스마트 관개 통제망(약 50.3억)과 고양도시관리공...,"[""고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영"", ""사단법인...","[""고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영"", ""한국철도...",0.500000
35,Q260,E,상,수쨔언꽁샤 씨엠애스 현장 건슐툥합고됴화 사업 쳥 에싼이랑요 그리구 고려댸 포탈시스템...,"[""고려대학교_차세대 포털·학사 정보시스템 구축사업"", ""한국수자원공사_건설통합시스...","[""고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고"", ""고려대학교_차세대...",0.500000



정답은 있으나 순위가 낮은 사례:


,id,type,difficulty,is_multi_doc,question,ground_truth_docs,retrieved_docs_top5,mrr_at_5,ndcg_at_5
8,Q132,B,중,True,인천공항운영서비스의 기존 ERP 시스템 고도화 사업과 경희대학교 산학협력단 정보시스...,"[""경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정"", ""인천공항운...","[""경희대학교_[재공고] 산학협력단 정보시스템 운영 용역업체 선정"", ""경희대학교_...",0.50,0.624051
33,Q453,B,상,True,"코이카-아시아물위원회 우즈벡 관개망, 한국연구재단 실태조사 시스템(1.293억), ...","[""부산관광공사_경영정보시스템 기능개선"", ""사단법인아시아물위원회사무국_우즈벡-키르...","[""KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원...",0.25,0.319147
34,Q473,B,중,True,아시아물위원회사무국 우즈베키스탄 스마트 관개 통제망(약 50.3억)과 고양도시관리공...,"[""고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영"", ""사단법인...","[""고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영"", ""한국철도...",1.00,0.613147
35,Q260,E,상,True,수쨔언꽁샤 씨엠애스 현장 건슐툥합고됴화 사업 쳥 에싼이랑요 그리구 고려댸 포탈시스템...,"[""고려대학교_차세대 포털·학사 정보시스템 구축사업"", ""한국수자원공사_건설통합시스...","[""고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고"", ""고려대학교_차세대...",0.50,0.386853
70,Q276,C,중,False,그럼 고려대학교 차세대 포털과 관련해서 질문할게요. 막대한 자본으로 모바일 친화적 ...,"[""고려대학교_차세대 포털·학사 정보시스템 구축사업""]","[""고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고"", ""고려대학교_차세대...",0.50,0.630930
73,Q376,C,중,False,아까 경희대학교 산학협력단 시스템 운영자가 해야 하는 추가 관리 업무들 설명해주셨죠...,"[""경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정""]","[""경희대학교산학협력단_산학협력단 정보시스템 운영 용역업체 선정"", ""경희대학교산학...",0.25,0.430677
82,Q257,D,하,False,고려대학교의 '차세대 포털·학사 정보시스템 구축' 제안서에서 사업 수주 업체 직원들...,"[""고려대학교_차세대 포털·학사 정보시스템 구축사업""]","[""고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고"", ""고려대학교_차세대...",0.50,0.630930
88,Q040,E,중,False,고대 차세대 ERP 말고 이번 학사 포털 플젝에서 메인으로 잡고 있는 레인지랑 타겟...,"[""고려대학교_차세대 포털·학사 정보시스템 구축사업""]","[""고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고"", ""고려대학교_차세대...",0.50,0.630930
90,Q099,E,하,False,꼬렫대헉교예셔 지냉하는 차쎄대 뽀털 학사 쩡보씨스탬 끄축샤업의 춍 에싼 규모를 좀 ...,"[""고려대학교_차세대 포털·학사 정보시스템 구축사업""]","[""국가평생교육진흥원_온라인 지식학습 플랫폼 보안 강화 및 백업시스템"", ""한국교육...",0.25,0.430677
93,Q160,E,상,False,인쳔공향 차세데 ERP 구츅 사옵에 춍 예산 얼마고 이거 사업하면 결국 운영진이 잴...,"[""인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 ERP시스템 구축""]","[""(재)경기도경제과학진흥원_경기도 기업SOS넷 개편 정보화전략(ISMP) 수립"",...",0.50,0.630930



질문 유형별 지표:


...

,chunk_type,count
0,table,2071
1,text,1156
2,fact_candidates,773



검색된 fact_type 분포:


,fact_type,count
0,budget,322
1,document_summary,165
2,business_type,102
3,duration,85
4,submission_documents,35
5,submission_logistics,24
6,bid_deadline,23
7,eligibility,17



저장 완료: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521/failure_focus_J5_hybrid_rrf_rerank.csv


,id,type,difficulty,is_multi_doc,failure_tags,question,ground_truth_docs,retrieved_docs_top5
0,Q054,B,상,True,multi_doc | partial_loss | 숫자/예산 질문 | 유사 문서/재공...,"한국가스공사의 '차세대 통합정보시스템', 나노종합기술원의 '설비온라인 시스템', 그...","[""그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역"", ""나노종...","[""그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역"", ""한국가..."
1,Q433,B,상,True,multi_doc | partial_loss | 숫자/예산 질문 | 유사 문서/재공...,"아시아물위원회사무국 우즈벡 관개 사업비, 부산관광공사 경영정보망 사업비, 광주문화재...","[""남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사"", ""부산...","[""부산관광공사_경영정보시스템 기능개선"", ""사단법인아시아물위원회사무국_우즈벡-키르..."
2,Q453,B,상,True,multi_doc | partial_loss | 숫자/예산 질문 | 유사 문서/재공...,"코이카-아시아물위원회 우즈벡 관개망, 한국연구재단 실태조사 시스템(1.293억), ...","[""부산관광공사_경영정보시스템 기능개선"", ""사단법인아시아물위원회사무국_우즈벡-키르...","[""KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원..."
3,Q473,B,중,True,multi_doc | partial_loss | 숫자/예산 질문 | 유사 문서/재공...,아시아물위원회사무국 우즈베키스탄 스마트 관개 통제망(약 50.3억)과 고양도시관리공...,"[""고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영"", ""사단법인...","[""고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영"", ""한국철도..."
4,Q260,E,상,True,multi_doc | partial_loss | 유사 문서/재공고/기관 alias ...,수쨔언꽁샤 씨엠애스 현장 건슐툥합고됴화 사업 쳥 에싼이랑요 그리구 고려댸 포탈시스템...,"[""고려대학교_차세대 포털·학사 정보시스템 구축사업"", ""한국수자원공사_건설통합시스...","[""고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고"", ""고려대학교_차세대..."
5,Q132,B,중,True,multi_doc | 표/요건 질문 | 유사 문서/재공고/기관 alias,인천공항운영서비스의 기존 ERP 시스템 고도화 사업과 경희대학교 산학협력단 정보시스...,"[""경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정"", ""인천공항운...","[""경희대학교_[재공고] 산학협력단 정보시스템 운영 용역업체 선정"", ""경희대학교_..."
9,Q276,C,중,False,유사 문서/재공고/기관 alias,그럼 고려대학교 차세대 포털과 관련해서 질문할게요. 막대한 자본으로 모바일 친화적 ...,"[""고려대학교_차세대 포털·학사 정보시스템 구축사업""]","[""고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고"", ""고려대학교_차세대..."
10,Q376,C,중,False,유사 문서/재공고/기관 alias,아까 경희대학교 산학협력단 시스템 운영자가 해야 하는 추가 관리 업무들 설명해주셨죠...,"[""경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정""]","[""경희대학교산학협력단_산학협력단 정보시스템 운영 용역업체 선정"", ""경희대학교산학..."
11,Q257,D,하,False,숫자/예산 질문 | 유사 문서/재공고/기관 alias,고려대학교의 '차세대 포털·학사 정보시스템 구축' 제안서에서 사업 수주 업체 직원들...,"[""고려대학교_차세대 포털·학사 정보시스템 구축사업""]","[""고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고"", ""고려대학교_차세대..."
12,Q040,E,중,False,유사 문서/재공고/기관 alias,고대 차세대 ERP 말고 이번 학사 포털 플젝에서 메인으로 잡고 있는 레인지랑 타겟...,"[""고려대학교_차세대 포털·학사 정보시스템 구축사업""]","[""고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고"", ""고려대학교_차세대..."


In [12]:
# 10. 질문별 상세 조회

# 상세 조회 출력량 설정
# 값이 너무 크면 VS Code/Colab 화면이 멈춘 것처럼 느려질 수 있습니다.
DETAIL_QUESTION_LIMIT = 3
DETAIL_CONTEXT_ROWS_PER_EXPERIMENT = 5

# 조회할 질문 ID 설정
QUESTION_IDS = failure_focus_df['id'].head(DETAIL_QUESTION_LIMIT).tolist()  # 실패/낮은 순위 질문 우선 조회

# QUESTION_IDS = analysis_df.head(5)['id'].tolist()  # 처음 질문부터 순서대로
# QUESTION_IDS = ['Q099', 'Q191', 'Q214']  # 특정 질문 직접 지정

print('상세 조회 질문 수:', len(QUESTION_IDS))
print('실험별 표시 context 수:', DETAIL_CONTEXT_ROWS_PER_EXPERIMENT)

for question_id in QUESTION_IDS:
    print('=' * 120)
    print('질문 ID:', question_id)
    base_row = all_results_df[all_results_df['id'].eq(question_id)].iloc[0]
    print('질문:', base_row['question'])
    print('정답 문서:', base_row['ground_truth_docs'])
    display(all_results_df[all_results_df['id'].eq(question_id)][[
        'experiment_id', 'retrieved_docs_top5', 'hit_at_5', 'mrr_at_5', 'ndcg_at_5',
        'doc_recall_at_5', 'multi_doc_recall_at_5', 'candidate_generation_failed_top10',
        'candidate_generation_failed_top30', 'partial_multi_doc_loss', 'retrieval_ms'
    ]])
    for exp_id in [exp.experiment_id for exp in experiments]:
        print('-' * 100)
        print(exp_id)
        show_df = all_contexts_df[(all_contexts_df['experiment_id'].eq(exp_id)) & (all_contexts_df['question_id'].eq(question_id))].copy()
        display(show_df[['rank', 'method', 'score', 'rerank_score', 'source_file', 'chunk_type', 'fact_type', 'section_path', 'text']].head(DETAIL_CONTEXT_ROWS_PER_EXPERIMENT))


상세 조회 질문 수: 3
실험별 표시 context 수: 5
질문 ID: Q054
질문: 한국가스공사의 '차세대 통합정보시스템', 나노종합기술원의 '설비온라인 시스템', 그랜드코리아레저의 '그룹웨어 시스템' 3개 인프라 전산화 구축 사업의 사업 목적을 공통 관점에서 살펴보았을 때, 이들이 근원적으로 동일하게 쟁점화하고 전환 지향하고 있는 '업무적 궁극적 개선 효과'가 무엇인지 짚고 해당 세 기관 중 가장 낮은 예산을 투입하는 기관명만 정확히 기재하십시오.
정답 문서: ["그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역", "나노종합기술원_스마트 팹 서비스 활용체계 구축관련 설비온라인 시스", "한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축"]


,experiment_id,retrieved_docs_top5,hit_at_5,mrr_at_5,ndcg_at_5,doc_recall_at_5,multi_doc_recall_at_5,candidate_generation_failed_top10,candidate_generation_failed_top30,partial_multi_doc_loss,retrieval_ms
4,J0_dense_baseline,"[""한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축"", ""그랜드코리아레저(...",1.0,1.0,0.765361,0.666667,0.666667,0,0,1,30.493641
104,J1_dense_wide,"[""한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축"", ""그랜드코리아레저(...",1.0,1.0,0.765361,0.666667,0.666667,0,0,1,36.960409
204,J2_bm25_only,"[""그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역"", ""대한상...",1.0,1.0,0.703918,0.666667,0.666667,0,0,1,220.581064
304,J3_dense_bm25_rrf,"[""그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역"", ""한국가...",1.0,1.0,0.765361,0.666667,0.666667,0,0,1,268.482072
404,J4_dense_rerank,"[""그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역"", ""한국가...",1.0,1.0,0.946902,1.000000,1.000000,0,0,0,2224.531125
504,J5_hybrid_rrf_rerank,"[""그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역"", ""한국가...",1.0,1.0,0.765361,0.666667,0.666667,0,0,1,3513.029164


----------------------------------------------------------------------------------------------------
J0_dense_baseline


,rank,method,score,rerank_score,source_file,chunk_type,fact_type,section_path,text
160,1,dense,-0.450746,NaN,한국가스공사_[재공고]차세대 통합정보시ᄉ...,text,,Ⅲ. 일반사항,[문서: 한국가스공사_[재공고]차세대 통합정...
161,2,dense,-0.497637,NaN,그랜드코리아레저(주)_2024년도 GKL 그룹웨어...,text,,문서 시작,사 업 명 : GKL 그룹웨어 시스템 구축사업 □ 사업기간 : 계약일로부터 6개월 ...
162,3,dense,-0.505122,NaN,국가철도공단_철도산업정보센터 정보시...,text,,Ⅵ. 제안서 세부목차 및 서식,"트, 운영시험)를 통한 시스템 안정성 확보 - DATA 전환 대상 : 통합자료관리,..."
163,4,dense,-0.522677,NaN,한국철도공사 (용역)_중장기 정보화전ᄅ...,text,,Ⅸ. 별지 서식,[문서: 한국철도공사 (용역)_중장기 정보ᄒ...
164,5,dense,-0.535773,NaN,한국가스공사_[재공고]차세대 통합정보시ᄉ...,text,,Ⅰ. 개요,[문서: 한국가스공사_[재공고]차세대 통합정...


----------------------------------------------------------------------------------------------------
J1_dense_wide


...

,rank,method,score,rerank_score,source_file,chunk_type,fact_type,section_path,text
25980,1,rrf,0.029551,NaN,KOICA 전자조달_[지문] [국제] (재공고)우즈...,text,,문서 시작,"$=1,290원, 2023년 기준환율) (부가가치세 등 제반 비용 포함) ㅇ 1차연..."
25981,2,rrf,0.016393,NaN,한국보건산업진흥원_의료기기산업 종ᄒ...,text,,Ⅲ. 제안요청 내용,[문서: 한국보건산업진흥원_의료기기산업...
25982,3,rrf,0.016129,NaN,국립중앙의료원_(긴급)「2024년도 차세대 ...,fact_candidates,budget,핵심 후보 정보 > budget,[문서: 국립중앙의료원_(긴급)「2024년도 차...
25983,4,rrf,0.016129,NaN,한국보건산업진흥원_의료기기산업 종ᄒ...,table,,Ⅲ. 제안요청 내용,[문서: 한국보건산업진흥원_의료기기산업...
25984,5,rrf,0.015873,NaN,한국보건산업진흥원_의료기기산업 종ᄒ...,fact_candidates,budget,핵심 후보 정보 > budget,[문서: 한국보건산업진흥원_의료기기산업...


----------------------------------------------------------------------------------------------------
J4_dense_rerank


,rank,method,score,rerank_score,source_file,chunk_type,fact_type,section_path,text
31320,1,dense_reranked,-0.591896,0.375722,KOICA 전자조달_[긴급] [지문] [국제] 우즈...,table,,문서 시작,[문서: KOICA 전자조달_[긴급] [지문] [국제]...
31321,2,dense_reranked,-0.563375,0.306953,KOICA_(재공고) 몽골 철도안전 통합관제...,text,,문서 시작,[문서: KOICA_(재공고) 몽골 철도안전 통합...
31322,3,dense_reranked,-0.558612,0.233794,KOICA 전자조달_[지문] [국제] (재공고)우즈...,text,,문서 시작,[문서: KOICA 전자조달_[지문] [국제] (재공고...
31323,4,dense_reranked,-0.578495,0.219395,부산관광공사_경영정보시스템 기능개선...,fact_candidates,budget,핵심 후보 정보 > budget,[문서: 부산관광공사_경영정보시스템 기능...
31324,5,dense_reranked,-0.595147,0.217810,KOICA_(재공고) 몽골 철도안전 통합관제...,table,,문서 시작,[문서: KOICA_(재공고) 몽골 철도안전 통합...


----------------------------------------------------------------------------------------------------
J5_hybrid_rrf_rerank


,rank,method,score,rerank_score,source_file,chunk_type,fact_type,section_path,text
35320,1,rrf_reranked,0.011364,0.375722,KOICA 전자조달_[긴급] [지문] [국제] 우즈...,table,,문서 시작,[문서: KOICA 전자조달_[긴급] [지문] [국제]...
35321,2,rrf_reranked,0.013889,0.306953,KOICA_(재공고) 몽골 철도안전 통합관제...,text,,문서 시작,[문서: KOICA_(재공고) 몽골 철도안전 통합...
35322,3,rrf_reranked,0.013889,0.250222,KOICA 전자조달_[지문] [국제] (재공고)우즈...,text,,문서 시작,"며, PMC(Project Management Consultant, 사업수행기관)에..."
35323,4,rrf_reranked,0.014706,0.233794,KOICA 전자조달_[지문] [국제] (재공고)우즈...,text,,문서 시작,[문서: KOICA 전자조달_[지문] [국제] (재공고...
35324,5,rrf_reranked,0.012500,0.219395,부산관광공사_경영정보시스템 기능개선...,fact_candidates,budget,핵심 후보 정보 > budget,[문서: 부산관광공사_경영정보시스템 기능...


## 공유 메모

- `RUN_MODE="exp100"`는 100문항 고정 샘플로 6개 retrieval 조건을 비교합니다.
- 주요 지표는 `hit_at_5_any`, `mrr_at_5`, `ndcg_at_5`, `doc_recall_at_5`를 골고루 봅니다. 단일문서는 `single_doc_mrr_at_5`, 다중문서는 `multi_doc_ndcg_at_5`, `multi_doc_recall_at_5`, `partial_multi_doc_loss`를 별도로 해석합니다.
- `candidate_generation_failed_top10`은 엄격한 후보 생성 실패 확인용이고, `candidate_generation_failed_top30`은 후보를 넓혔을 때도 정답이 없는지 보는 참고용입니다.
- Chroma DB, 임베딩 캐시, 실행 결과 파일은 기본적으로 GitHub에 올리지 않습니다.
- 결과 공유 시에는 `experiment_summary_exp100.csv`, `failure_focus_*.csv`, 필요한 실험별 results CSV만 공유하면 됩니다.

In [13]:
from pathlib import Path

out = Path("/content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/retrieval_quickcheck_p4_hwpx_125/exp100_0521")

for name in [
    "experiment_summary_exp100_0521.csv",
    "all_experiment_results_exp100_0521.csv",
    "all_experiment_contexts_exp100_0521.jsonl",
    "failure_focus_J5_hybrid_rrf_rerank.csv",
]:
    p = out / name
    print(name, p.exists(), f"{p.stat().st_size / 1024 / 1024:.2f} MiB" if p.exists() else "")

experiment_summary_exp100_0521.csv True 0.00 MiB
all_experiment_results_exp100_0521.csv True 1.76 MiB
all_experiment_contexts_exp100_0521.jsonl True 54.67 MiB
failure_focus_J5_hybrid_rrf_rerank.csv True 0.05 MiB
